# FiLM Gamma/Beta Projector
FiLM: gamma * x + beta.

In [1]:
import sys
sys.path.insert(0, '..')
import torch, os
from gensim.models import KeyedVectors
from src.data import load_data
from src.models import build_model
from src.train import train
from src.evaluate import run_all_evaluations
from src.visualize import plot_curves, visualize_3d_projection
os.makedirs('../results/film_gamma_beta', exist_ok=True)

In [2]:
embeddings = KeyedVectors.load_word2vec_format(
    '../gensim-data/glove-wiki-gigaword-300/glove-wiki-gigaword-300.gz'
)
print(f'Vocab size: {len(embeddings.key_to_index)}')

Vocab size: 400000


In [3]:
train_loader, val_loader, gender_clean, royalty_clean = load_data(embeddings)


--------------------------------------------------
  GENDER PAIRS
--------------------------------------------------
  candidates: 100 | valid ones: 100 | discarded: 0

--------------------------------------------------
  ROYALTY PAIRS
--------------------------------------------------
  candidates: 106 | valid ones: 106 | discarded: 0

  analysis: GENDER
  cos mean=0.4609 std=0.1898 | threshold 0.3: 83/100

  analysis: STATUS
  cos mean=0.2943 std=0.1196 | threshold 0.3: 51/106

  Cosine between axes: 0.3172

  Train: 106 | Val: 28


In [4]:
model = build_model('film_gamma_beta')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model, history = train(
    projector=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, lambda_ortho=0.1, num_epochs=1500,
)
torch.save(model.state_dict(), '../results/film_gamma_beta/model.pt')

Training:   0%|          | 0/1500 [00:00<?, ?it/s]


Best model: epoch 1498, val_loss=0.0704


In [5]:
plot_curves(history)

In [6]:
metrics = run_all_evaluations(model, embeddings)
print(metrics)

Analigies:
  king - man + woman → queen  |  cos=0.5878  dist=1.0421
  prince - boy + girl → princess  |  cos=0.9997  dist=0.2465
  husband - man + woman → wife  |  cos=0.4324  dist=0.4425
  uncle - man + woman → aunt  |  cos=0.3091  dist=0.4212
  father - man + woman → mother  |  cos=0.8001  dist=0.3576
  brother - man + woman → sister  |  cos=0.0243  dist=0.8523
  king - prince + princess → queen  |  cos=0.7020  dist=1.0340
  grandfather - father + mother → grandmother  |  cos=0.9613  dist=0.2243
  actor - man + woman → actress  |  cos=0.6661  dist=0.5643
  he - man + woman → she  |  cos=0.5648  dist=0.6034
  analogy_cos=0.6048 | analogy_dist=0.5788
{'analogy_cos': 0.6047652055189946, 'analogy_dist': 0.5788210034370422}


In [8]:
for axis_name, condition in [('gender', [1,0,0]), ('status', [0,1,0])]:
    visualize_3d_projection(
        gender_pairs=gender_clean, royalty_pairs=royalty_clean,
        embeddings=embeddings, projector=model,
        condition=condition,
        output_html=f'../results/film_gamma_beta/film_gamma_beta_{axis_name}_3d.html',
    )

Saved: ../results/film_gamma_beta/film_gamma_beta_gender_3d.html


Saved: ../results/film_gamma_beta/film_gamma_beta_status_3d.html
